In [ ]:
# Imports and configuration
import os
import math
import json
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs

# Backend base URL
backendUrl = os.getenv('CASE_MGMT_BACKEND_URL', 'http://localhost:3090')

query_string = os.environ.get("QUERY_STRING", "")
params = parse_qs(query_string)

# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}

# Fallback counterparty ID (verified working one)
FALLBACK_CP_ID = 'dbtr_b22cd4cd61b9407d825f40be29bb5fe9'

# Counterparty to visualize
counterpartyId = params.get('entityId')[0]

# print(f'Using backend: {backendUrl}')
# print(f'Using counterpartyId: {counterpartyId}')

In [ ]:
# Fetch counterparty network analysis data
def fetch_counterparty_network(cp_id, backend_url):
  
    url = f"{backend_url}/api/v1/jupyter/proxy/network-analysis/counterparty-node/{cp_id}"
    params = {'tenantId': 'DEFAULT'}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        if resp.status_code == 404 and cp_id != FALLBACK_CP_ID:
            print(f'Network data not found for {cp_id}, trying fallback: {FALLBACK_CP_ID}')
            url = f"{backend_url}/api/v1/jupyter/proxy/network-analysis/counterparty-node/{FALLBACK_CP_ID}"
            resp = requests.get(url, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print('Error fetching counterparty network:', e)
        return None

data = fetch_counterparty_network(counterpartyId, backendUrl)
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for counterparty: {counterpartyId}</div>"))
# else:
#     print('Fetched network data keys:', list(data.keys()))
#     network = data.get('network', {})
#     print(f'Nodes: {len(network.get("nodes", []))}, Edges: {len(network.get("edges", []))})')

In [ ]:
# Normalise network payloads
nodes = []
edges = []

if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        nodes = network_data.get('nodes', [])
        edges = network_data.get('edges', [])
    else:
        nodes = data.get('nodes', [])
        edges = data.get('edges', [])

nodes = [n for n in nodes if isinstance(n, dict)]
edges = [e for e in edges if isinstance(e, dict)]
counterparty_details = data.get('counterpartyDetails', {}) if isinstance(data, dict) else {}

# Get the root node ID (center counterparty) from the network data
root_node_id = None
if data and isinstance(data, dict):
    network_data = data.get('network')
    if network_data and isinstance(network_data, dict):
        root_node_id = network_data.get('rootNodeId')

if not root_node_id:
    root_node_id = counterpartyId

# Helper functions to extract flags
def is_node_alerted(node):
    flags = node.get('flags', {})
    return flags.get('alerted', False)

def is_node_investigated(node):
    flags = node.get('flags', {})
    return flags.get('investigated', False)

def is_edge_alerted(edge):
    flags = edge.get('flags', {})
    return flags.get('alerted', False)

def is_edge_investigated(edge):
    flags = edge.get('flags', {})
    return flags.get('investigated', False)

def get_node_size(node, root_node_id):
    if node.get('id') == root_node_id:
        return 95
    if is_node_alerted(node):
        return 80
    return 75

size_map = {n.get('id'): get_node_size(n, root_node_id) for n in nodes if n.get('id')}

def build_positions(nodes, root_node_id):
    positions = {}
    n = max(len(nodes), 1)
    center_idx = 0
    for i, node in enumerate(nodes):
        if node.get('id') == root_node_id:
            center_idx = i
            break
    radius = 280
    angle_step = 2 * math.pi / max(n - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1
    return positions

positions = build_positions(nodes, root_node_id)

def offset_edge(x0, y0, x1, y1, start_offset, end_offset):
    dx = x1 - x0
    dy = y1 - y0
    dist = math.hypot(dx, dy)
    if dist == 0:
        return x0, y0, x1, y1
    ux = dx / dist
    uy = dy / dist
    return (x0 + ux * start_offset, y0 + uy * start_offset,
            x1 - ux * end_offset, y1 - uy * end_offset)

# Build edge segments
edge_segments = {
    'alert': {'x': [], 'y': []},
    'investigation': {'x': [], 'y': []},
    'normal': {'x': [], 'y': []}
}

for e in edges:
    s = e.get('source') or e.get('from')
    t = e.get('target') or e.get('to')
    if s in positions and t in positions:
        x0, y0 = positions[s]
        x1, y1 = positions[t]
        so = (size_map.get(s, 75) / 2) + 6
        eo = (size_map.get(t, 75) / 2) + 6
        x0, y0, x1, y1 = offset_edge(x0, y0, x1, y1, so, eo)
        if is_edge_alerted(e):
            key = 'alert'
        elif is_edge_investigated(e):
            key = 'investigation'
        else:
            key = 'normal'
        edge_segments[key]['x'] += [x0, x1, None]
        edge_segments[key]['y'] += [y0, y1, None]

edge_traces = [
    go.Scatter(x=edge_segments['normal']['x'], y=edge_segments['normal']['y'],
               mode='lines', line=dict(width=2, color='#8b5cf6', dash='solid'),
               hoverinfo='none', showlegend=False),
    go.Scatter(x=edge_segments['alert']['x'], y=edge_segments['alert']['y'],
               mode='lines', line=dict(width=2, color='#ef4444', dash='solid'),
               hoverinfo='none', showlegend=False),
    go.Scatter(x=edge_segments['investigation']['x'], y=edge_segments['investigation']['y'],
               mode='lines', line=dict(width=2, color='#f59e0b', dash='solid'),
               hoverinfo='none', showlegend=False),
]

# Build node traces
node_x, node_y, node_text, node_labels = [], [], [], []
marker_size, marker_color, marker_line_color, marker_line_width, node_ids = [], [], [], [], []
ring_x, ring_y, ring_sizes = [], [], []

def format_display_id(value):
    if isinstance(value, str) and not value.upper().startswith('CP-') and len(value) >= 4:
        return f"CP-{value[-4:]}"
    return value or '-'

for n in nodes:
    nid = n.get('id')
    if nid not in positions:
        continue
    x, y = positions[nid]
    node_x.append(x)
    node_y.append(y)
    display_id = format_display_id(nid)
    is_center = nid == root_node_id

    if is_center and counterparty_details.get('name'):
        name = counterparty_details['name']
        short = name if len(name) <= 14 else f"{name[:12]}…"
        node_labels.append(f"{display_id}<br>{short}")
        node_text.append(f"<b>{name}</b><br>ID: {nid}")
    else:
        label = n.get('label') or ''
        short = label if len(label) <= 14 else f"{label[:12]}…"
        node_labels.append(f"{display_id}<br>{short}" if short else display_id)
        node_text.append(f"<b>Counterparty</b><br>ID: {nid}")

    is_alerted = is_node_alerted(n)
    is_investigated = is_node_investigated(n)

    if is_center:
        color, lc, sz, lw = '#ede9fe', '#8b5cf6', 95, 3
    elif is_alerted:
        color, lc, sz, lw = '#fee2e2', '#ef4444', 80, 2
    else:
        color, lc, sz, lw = '#dbeafe', '#3b82f6', 75, 2

    if is_investigated:
        ring_x.append(x)
        ring_y.append(y)
        ring_sizes.append(sz + 12)

    marker_color.append(color)
    marker_line_color.append(lc)
    marker_line_width.append(lw)
    marker_size.append(sz)
    node_ids.append(nid)

ring_trace = go.Scatter(
    x=ring_x, y=ring_y, mode='markers',
    marker=dict(size=ring_sizes, color='rgba(0,0,0,0)', line=dict(width=3, color='#f59e0b')),
    hoverinfo='none', showlegend=False)

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    text=node_labels, hovertext=node_text, hoverinfo='text', customdata=node_ids,
    marker=dict(color=marker_color, size=marker_size,
                line=dict(width=marker_line_width, color=marker_line_color)),
    textposition='middle center',
    textfont=dict(size=9, color='#1e293b', family='Arial, sans-serif'),
    showlegend=False)

# Stats helpers
def get_edge_count(e):
    return e.get('txCount') or e.get('transactionCount') or e.get('count') or 1

def get_edge_amount(e):
    return e.get('totalAmount') or e.get('amount') or e.get('totalValue') or 0

def stats_for_node(node_id):
    tx, total = 0, 0
    for e in edges:
        s = e.get('source') or e.get('from')
        t = e.get('target') or e.get('to')
        if s == node_id or t == node_id:
            tx += get_edge_count(e)
            total += get_edge_amount(e)
    return tx, total

# Initial sidebar values from center node
initial_node_id = root_node_id or (node_ids[0] if node_ids else None)
initial_tx, initial_total = stats_for_node(initial_node_id) if initial_node_id else (0, 0)
initial_node = next((n for n in nodes if n.get('id') == initial_node_id), {})
initial_is_center = bool(initial_node.get('id') == root_node_id)

initial_cp_id = counterparty_details.get('counterpartyId') if initial_is_center else initial_node.get('id')
initial_name = (counterparty_details.get('name') if initial_is_center else initial_node.get('label')) or 'null'
initial_relationship = (
    'Center Node' if initial_is_center
    else (initial_node.get('relationship') or initial_node.get('relationshipType') or 'null')
)
initial_velocity = (
    counterparty_details.get('velocity') if initial_is_center
    else (initial_node.get('velocity') or 'null')
) or 'null'

if initial_is_center:
    initial_tx = counterparty_details.get('transactions', initial_tx)
    initial_total = counterparty_details.get('totalValue', initial_total)

cp_flags = counterparty_details.get('flags', {})
initial_is_alert = cp_flags.get('alerted', False) if initial_is_center else is_node_alerted(initial_node)
initial_is_investigation = cp_flags.get('investigated', False) if initial_is_center else is_node_investigated(initial_node)

fig = go.Figure(
    data=edge_traces + [ring_trace, node_trace],
    layout=go.Layout(
        title=None, showlegend=False, hovermode='closest', dragmode=False,
        margin=dict(b=20, l=20, r=20, t=20),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-380, 380]),
        height=560, paper_bgcolor='#ffffff', plot_bgcolor='#ffffff'
    )
)

if nodes:
    graph_id = 'counterparty-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False, 'scrollZoom': False}, div_id=graph_id)

    legend_html = """
    <div style="position:absolute;bottom:12px;left:12px;background:white;padding:10px;border-radius:8px;border:1px solid #e2e8f0;box-shadow:0 4px 10px rgba(15,23,42,0.08);font-family:Arial,sans-serif;font-size:10px;">
        <div style="font-weight:600;margin-bottom:8px;color:#0f172a;font-size:11px;">Legend</div>
        <div style="display:flex;align-items:center;margin-bottom:6px;">
            <div style="width:12px;height:12px;border-radius:50%;background:#fee2e2;border:2px solid #ef4444;margin-right:6px;"></div>
            <span style="color:#475569;">Alert Triggered</span>
        </div>
        <div style="display:flex;align-items:center;margin-bottom:6px;">
            <div style="width:12px;height:12px;border-radius:50%;background:#dbeafe;border:2px solid #3b82f6;margin-right:6px;"></div>
            <span style="color:#475569;">Normal Counterparty</span>
        </div>
        <div style="display:flex;align-items:center;margin-bottom:6px;">
            <div style="width:12px;height:12px;border-radius:50%;background:#ede9fe;border:2px solid #8b5cf6;margin-right:6px;box-shadow:0 0 0 2px #f59e0b;"></div>
            <span style="color:#475569;">Under Investigation</span>
        </div>
        <div style="display:flex;align-items:center;margin-bottom:4px;">
            <div style="width:20px;height:2px;background:#8b5cf6;margin-right:6px;"></div>
            <span style="color:#475569;">1st Degree</span>
        </div>
        <div style="display:flex;align-items:center;">
            <div style="width:20px;height:0;border-top:2px dashed #9ca3af;margin-right:6px;"></div>
            <span style="color:#475569;">2nd Degree</span>
        </div>
    </div>
    """

    zoom_controls_html = """
    <div style="position:absolute;top:16px;right:16px;background:white;padding:8px;border-radius:10px;border:1px solid #e2e8f0;box-shadow:0 4px 10px rgba(15,23,42,0.08);display:flex;flex-direction:column;gap:8px;">
        <button id="cp-zoom-in-btn" style="width:34px;height:34px;border:1px solid #e2e8f0;background:#ffffff;border-radius:6px;cursor:pointer;display:flex;align-items:center;justify-content:center;color:#1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><circle cx="11" cy="11" r="8"/><line x1="11" y1="8" x2="11" y2="14"/><line x1="8" y1="11" x2="14" y2="11"/><line x1="21" y1="21" x2="16.65" y2="16.65"/></svg>
        </button>
        <button id="cp-zoom-out-btn" style="width:34px;height:34px;border:1px solid #e2e8f0;background:#ffffff;border-radius:6px;cursor:pointer;display:flex;align-items:center;justify-content:center;color:#1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><circle cx="11" cy="11" r="8"/><line x1="8" y1="11" x2="14" y2="11"/><line x1="21" y1="21" x2="16.65" y2="16.65"/></svg>
        </button>
        <button id="cp-reset-zoom-btn" style="width:34px;height:34px;border:1px solid #e2e8f0;background:#ffffff;border-radius:6px;cursor:pointer;display:flex;align-items:center;justify-content:center;color:#1e293b;" onmouseover="this.style.background='#f1f5f9'" onmouseout="this.style.background='#ffffff'" title="Reset Zoom">
            <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M3 12a9 9 0 0 1 9-9 9.75 9.75 0 0 1 6.74 2.74L21 8"/><path d="M21 3v5h-5"/><path d="M21 12a9 9 0 0 1-9 9 9.75 9.75 0 0 1-6.74-2.74L3 16"/><path d="M3 21v-5h5"/></svg>
        </button>
    </div>
    """

    freq_color = '#ef4444' if initial_velocity == 'HIGH' else '#22c55e'
    alert_display = 'block' if initial_is_alert else 'none'
    inv_display = 'block' if initial_is_investigation else 'none'
    total_formatted = '${:,.2f}'.format(initial_total)

    sidebar_html = f"""
    <div id="cp-sidebar-panel" style="width:300px;background:white;padding:24px;font-family:Arial,sans-serif;">
        <div style="margin-bottom:18px;">
            <div style="font-size:18px;font-weight:700;color:#0f172a;margin-bottom:12px;">Counterparty Details</div>
            <div style="margin-bottom:10px;">
                <div style="font-size:10px;color:#64748b;text-transform:uppercase;margin-bottom:4px;">COUNTERPARTY ID</div>
                <div id="cp-sidebar-id" style="font-size:13px;color:#0f172a;font-weight:600;word-break:break-all;">{initial_cp_id or '-'}</div>
            </div>
            <div id="cp-relationship-row" style="margin-bottom:10px;">
                <div style="font-size:10px;color:#64748b;text-transform:uppercase;margin-bottom:4px;">RELATIONSHIP</div>
                <div id="cp-sidebar-relationship" style="font-size:12px;color:#0f172a;font-weight:600;">{initial_relationship}</div>
            </div>
        </div>
        <div style="height:1px;background:#e2e8f0;margin:16px 0 18px 0;"></div>
        <div>
            <div style="font-size:11px;font-weight:700;color:#64748b;margin-bottom:12px;text-transform:uppercase;">Transaction Statistics</div>
            <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
                <span style="color:#64748b;font-size:12px;">Transactions:</span>
                <span id="cp-stat-total-tx" style="font-weight:600;color:#0f172a;">{initial_tx}</span>
            </div>
            <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
                <span style="color:#64748b;font-size:12px;">Total Value:</span>
                <span id="cp-stat-total-value" style="font-weight:600;color:#0f172a;">{total_formatted}</span>
            </div>
            <div style="display:flex;justify-content:space-between;">
                <span style="color:#64748b;font-size:12px;">Frequency:</span>
                <span id="cp-stat-frequency" style="font-weight:700;color:{freq_color};">{initial_velocity}</span>
            </div>
        </div>
        <div id="cp-alert-box" style="margin-top:16px;padding:10px 12px;border:1px solid #fecaca;background:#fef2f2;color:#dc2626;border-radius:8px;font-size:12px;font-weight:600;display:{alert_display};">
            ⚠ Alert triggered on this counterparty
        </div>
        <div id="cp-investigation-box" style="margin-top:10px;padding:10px 12px;border:1px solid #fde68a;background:#fffbeb;color:#b45309;border-radius:8px;font-size:12px;font-weight:600;display:{inv_display};">
            🔍 Currently under investigation
        </div>
    </div>
    """

    interactive_script = f"""
    <script>
    (function() {{
        var nodes = {json.dumps(nodes)};
        var edges = {json.dumps(edges)};
        var cpDetails = {json.dumps(counterparty_details)};
        var rootNodeId = {json.dumps(root_node_id)};
        var originalPositions = {json.dumps(positions)};
        var graphDiv = document.getElementById('{graph_id}');
        
        var currentZoomLevel = 1.0;
        var baseRange = 380;
        var minZoom = 0.3;
        var maxZoom = 3.0;

        var numberFormatter = new Intl.NumberFormat();
        var currencyFormatter = new Intl.NumberFormat(undefined, {{
            style: 'currency', currency: 'USD',
            minimumFractionDigits: 2, maximumFractionDigits: 2
        }});

        function getFrequencyColor(freq) {{
            if (!freq) return '#22c55e';
            var f = freq.toUpperCase();
            if (f === 'HIGH') return '#ef4444';
            if (f === 'MEDIUM') return '#f59e0b';
            return '#22c55e';
        }}

        function getEdgeCount(e) {{
            return e.txCount || e.transactionCount || e.count || 1;
        }}

        function getEdgeAmount(e) {{
            return e.totalAmount || e.amount || e.totalValue || 0;
        }}

        function statsForNode(nodeId) {{
            var tx = 0, total = 0;
            edges.forEach(function(e) {{
                var src = e.source || e.from;
                var tgt = e.target || e.to;
                if (src === nodeId || tgt === nodeId) {{
                    tx += getEdgeCount(e);
                    total += getEdgeAmount(e);
                }}
            }});
            return {{ tx: tx, total: total }};
        }}

        function getScaledPositions() {{
            var scaledPos = {{}};
            for (var nodeId in originalPositions) {{
                var pos = originalPositions[nodeId];
                scaledPos[nodeId] = [pos[0] * currentZoomLevel, pos[1] * currentZoomLevel];
            }}
            return scaledPos;
        }}

        function getNodeSize(node) {{
            if (node.id === rootNodeId) return 95;
            var flags = node.flags || {{}};
            if (flags.alerted) return 80;
            return 75;
        }}

        function offsetEdge(x0, y0, x1, y1, startOffset, endOffset) {{
            var dx = x1 - x0;
            var dy = y1 - y0;
            var dist = Math.hypot(dx, dy);
            if (dist === 0) return [x0, y0, x1, y1];
            var ux = dx / dist;
            var uy = dy / dist;
            return [
                x0 + ux * startOffset,
                y0 + uy * startOffset,
                x1 - ux * endOffset,
                y1 - uy * endOffset
            ];
        }}

        function rebuildGraph() {{
            var positions = getScaledPositions();
            var sizeMap = {{}};
            var baseSizes = {{}};
            
            // Calculate base and scaled sizes
            nodes.forEach(function(n) {{
                var baseSize = getNodeSize(n);
                baseSizes[n.id] = baseSize;
                sizeMap[n.id] = baseSize * currentZoomLevel;
            }});

            // Rebuild edges
            var edgeSegments = {{
                alert: {{ x: [], y: [] }},
                investigation: {{ x: [], y: [] }},
                normal: {{ x: [], y: [] }}
            }};

            edges.forEach(function(e) {{
                var s = e.source || e.from;
                var t = e.target || e.to;
                if (positions[s] && positions[t]) {{
                    var x0 = positions[s][0], y0 = positions[s][1];
                    var x1 = positions[t][0], y1 = positions[t][1];
                    
                    var startOffset = (sizeMap[s] || 75) / 2 + 6;
                    var endOffset = (sizeMap[t] || 75) / 2 + 6;
                    var coords = offsetEdge(x0, y0, x1, y1, startOffset, endOffset);
                    
                    var flags = e.flags || {{}};
                    var key = 'normal';
                    if (flags.alerted) key = 'alert';
                    else if (flags.investigated) key = 'investigation';
                    
                    edgeSegments[key].x.push(coords[0], coords[2], null);
                    edgeSegments[key].y.push(coords[1], coords[3], null);
                }}
            }});

            // Rebuild nodes
            var nodeX = [], nodeY = [], ringX = [], ringY = [];
            var nodeSizes = [], ringSizes = [], nodeLineWidths = [];

            nodes.forEach(function(n) {{
                var nid = n.id;
                if (positions[nid]) {{
                    var x = positions[nid][0], y = positions[nid][1];
                    nodeX.push(x);
                    nodeY.push(y);
                    nodeSizes.push(sizeMap[nid]);
                    
                    var isCenter = nid === rootNodeId;
                    var baseLineWidth = isCenter ? 3 : 2;
                    nodeLineWidths.push(baseLineWidth * currentZoomLevel);

                    var flags = n.flags || {{}};
                    if (isCenter || flags.investigated) {{
                        ringX.push(x);
                        ringY.push(y);
                        ringSizes.push(sizeMap[nid] + 12 * currentZoomLevel);
                    }}
                }}
            }});

            // Update all traces with positions and sizes
            Plotly.restyle(graphDiv, {{
                'x': [edgeSegments.normal.x, edgeSegments.alert.x, edgeSegments.investigation.x, ringX, nodeX],
                'y': [edgeSegments.normal.y, edgeSegments.alert.y, edgeSegments.investigation.y, ringY, nodeY],
                'marker.size': [undefined, undefined, undefined, ringSizes, nodeSizes],
                'line.width': [2 * currentZoomLevel, 2 * currentZoomLevel, 2 * currentZoomLevel, undefined, undefined],
                'marker.line.width': [undefined, undefined, undefined, 3 * currentZoomLevel, nodeLineWidths],
                'textfont.size': [undefined, undefined, undefined, undefined, 9 * currentZoomLevel]
            }}, [0, 1, 2, 3, 4]);

            // Update axis ranges to fit the zoomed graph
            var dynamicRange = baseRange * Math.max(1, currentZoomLevel);
            Plotly.relayout(graphDiv, {{
                'xaxis.range': [-dynamicRange, dynamicRange],
                'yaxis.range': [-dynamicRange, dynamicRange]
            }});
        }}

        function zoomIn() {{
            var newZoom = currentZoomLevel * 1.3;
            if (newZoom <= maxZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function zoomOut() {{
            var newZoom = currentZoomLevel / 1.3;
            if (newZoom >= minZoom) {{
                currentZoomLevel = newZoom;
                rebuildGraph();
                updateZoomButtons();
            }}
        }}

        function resetZoom() {{
            currentZoomLevel = 1.0;
            rebuildGraph();
            updateZoomButtons();
        }}

        function updateZoomButtons() {{
            var zoomInBtn = document.getElementById('cp-zoom-in-btn');
            var zoomOutBtn = document.getElementById('cp-zoom-out-btn');
            var resetBtn = document.getElementById('cp-reset-zoom-btn');
            
            if (currentZoomLevel * 1.3 > maxZoom) {{
                zoomInBtn.style.opacity = '0.4';
                zoomInBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomInBtn.style.opacity = '1';
                zoomInBtn.style.cursor = 'pointer';
            }}
            
            if (currentZoomLevel / 1.3 < minZoom) {{
                zoomOutBtn.style.opacity = '0.4';
                zoomOutBtn.style.cursor = 'not-allowed';
            }} else {{
                zoomOutBtn.style.opacity = '1';
                zoomOutBtn.style.cursor = 'pointer';
            }}
            
            var zoomPercent = Math.round(currentZoomLevel * 100);
            resetBtn.setAttribute('title', 'Reset Zoom (current: ' + zoomPercent + '%)');
        }}

        function updateSidebar(nodeId) {{
            var node = nodes.find(function(n) {{ return n.id === nodeId; }}) || {{}};
            var isCenter = nodeId === rootNodeId;
            var stats = statsForNode(nodeId);

            var cpId = isCenter ? (cpDetails.counterpartyId || node.id) : node.id;
            var cpName = isCenter ? (cpDetails.name || node.label || 'null') : (node.label || 'null');
            var relationship = isCenter ? 'Center Node' : (node.relationship || node.relationshipType || 'null');

            document.getElementById('cp-sidebar-id').innerText = cpId || 'null';
            document.getElementById('cp-sidebar-name').innerText = cpName;
            document.getElementById('cp-sidebar-relationship').innerText = relationship;

            var txCount = stats.tx || 0;
            var totalValue = stats.total || 0;
            var frequency = node.velocity || 'null';
            if (isCenter && cpDetails) {{
                txCount = cpDetails.transactions || txCount;
                totalValue = cpDetails.totalValue || totalValue;
                frequency = cpDetails.velocity || node.velocity || 'null';
            }}

            document.getElementById('cp-stat-total-tx').innerText = numberFormatter.format(txCount);
            document.getElementById('cp-stat-total-value').innerText = currencyFormatter.format(totalValue);

            var freqEl = document.getElementById('cp-stat-frequency');
            freqEl.innerText = frequency;
            freqEl.style.color = getFrequencyColor(frequency);

            var detailFlags = cpDetails.flags || {{}};
            var nodeFlags = node.flags || {{}};
            var isAlert = isCenter ? !!detailFlags.alerted : !!nodeFlags.alerted;
            var isInvestigation = isCenter ? !!detailFlags.investigated : !!nodeFlags.investigated;

            document.getElementById('cp-alert-box').style.display = isAlert ? 'block' : 'none';
            document.getElementById('cp-investigation-box').style.display = isInvestigation ? 'block' : 'none';
        }}

        // Add mousewheel zoom support
        if (graphDiv) {{
            graphDiv.addEventListener('wheel', function(e) {{
                e.preventDefault();
                if (e.deltaY < 0) {{
                    zoomIn();
                }} else {{
                    zoomOut();
                }}
            }}, {{ passive: false }});

            graphDiv.on('plotly_click', function(d) {{
                if (d.points && d.points[0] && d.points[0].customdata) {{
                    updateSidebar(d.points[0].customdata);
                }}
            }});
        }}

        document.getElementById('cp-zoom-in-btn').addEventListener('click', zoomIn);
        document.getElementById('cp-zoom-out-btn').addEventListener('click', zoomOut);
        document.getElementById('cp-reset-zoom-btn').addEventListener('click', resetZoom);
        
        updateZoomButtons();
        updateSidebar(rootNodeId);
    }})();
    </script>
    """

    container_html = f"""
    <div style="display:flex;background:#ffffff;border:1px solid #e2e8f0;border-radius:12px;overflow:hidden;box-shadow:0 6px 16px rgba(15,23,42,0.08);">
        <div style="flex:1;background:white;position:relative;">
            {graph_html}
            {legend_html}
            {zoom_controls_html}
        </div>
        <div style="width:1px;background:#e2e8f0;"></div>
        {sidebar_html}
    </div>
    {interactive_script}
    """

    display(HTML(container_html))
else:
    display(HTML('<div style="color:#ef4444;padding:20px;">No counterparty network data available</div>'))